**Identifying the T3 countries for each country considered as the OL**

In [ ]:
from data_processing import DataSetup

flight_matrix_path = 'pgfgleam/datasets/country_flow_202001 1.csv'
country_codes_path = 'pgfgleam/datasets/geom_countries_codes.csv'
population_path = 'pgfgleam/datasets/WPP2019_TotalPopulation2020.csv'

flight_matrix, normalized_flow_matrix, source_countries, target_countries, source_codes, target_codes, population, name_mapping = DataSetup(flight_matrix_path, population_path, country_codes_path).load_and_process_data().values()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.sparse import csr_array

import warnings
warnings.filterwarnings("ignore")
import os


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import ast
import re
from matplotlib.patches import Rectangle

# Helper function to parse arrays safely
def safe_parse_array(x):
    if pd.isna(x) or x is None:
        return np.array([])
    if isinstance(x, str):
        if x.strip() == 'Float64[]' or x.strip() == '[]':
            return np.array([])
        x = re.sub(r'Float64\[', '[', x)
        try:
            return np.array(ast.literal_eval(x))
        except:
            return np.array([])
    return x

# Load the dataset
df = pd.read_csv('pgfgleam/all_results/local/full_results_selected_countries.csv')

print("Dataset columns:", df.columns.tolist())
print("Dataset shape:", df.shape)
print("Unique countries:", df['country'].unique())
print("Unique R0 values:", df['R0'].unique())
print("Unique gen_time values:", df['gen_time'].unique())

# Parse the array columns for WW (base_pdet=0.16, 10% sampling = 1.6%)
df['WW_detection_times_parsed'] = df['WW_016_50pct_detection_times'].apply(safe_parse_array)
df['WW_local_cases_samples_parsed'] = df['WW_016_50pct_local_cases_samples'].apply(safe_parse_array)
df['ICU_detection_times_parsed'] = df['ICU_detection_times'].apply(safe_parse_array)
df['ICU_local_cases_samples_parsed'] = df['ICU_local_cases_samples'].apply(safe_parse_array)

# Remove rows with empty arrays
df_valid = df[
    (df['WW_local_cases_samples_parsed'].apply(lambda x: len(x) > 0)) &
    (df['ICU_local_cases_samples_parsed'].apply(lambda x: len(x) > 0))
].copy()

print(f"\nAfter filtering: {len(df_valid)} rows with valid data")

# Get unique parameter combinations
param_combinations = df_valid[['R0', 'gen_time']].drop_duplicates().sort_values(['R0', 'gen_time'])
print(f"\nParameter combinations to plot: {len(param_combinations)}")
print(param_combinations)

# Select three countries to plot (excluding one)
all_countries = df_valid['country'].unique()
print(f"\nAll countries available: {all_countries}")

# You can modify this list to choose which 3 countries you want
countries_to_plot = [all_countries[2], all_countries[0], all_countries[1]]
print(f"Countries selected for plotting: {countries_to_plot}")

# Define colors (turquoise for WW, orange for ICU)
color_ww = '#4FB3BE'  # Turquoise
color_icu = '#E87D47'  # Orange

# Create a plot for each (R0, gen_time) combination
for idx, (_, param_row) in enumerate(param_combinations.iterrows()):
    R0_val = param_row['R0']
    gen_time_val = param_row['gen_time']
    
    print(f"\n{'='*60}")
    print(f"Plotting R0={R0_val}, gen_time={gen_time_val}")
    print(f"{'='*60}")
    
    # Filter data for this parameter combination
    param_data = df_valid[
        (df_valid['R0'] == R0_val) & 
        (df_valid['gen_time'] == gen_time_val) &
        (df_valid['country'].isin(countries_to_plot))
    ].copy()
    
    if len(param_data) == 0:
        print(f"No data for R0={R0_val}, gen_time={gen_time_val}")
        continue
    
    print(f"Countries found: {param_data['country'].tolist()}")
    
    # Create figure with 2 panels
    fig, (ax4, ax3) = plt.subplots(1, 2, figsize=(20, 7))
    fig.subplots_adjust(wspace=-1.0)
    
    # =============== PLOT 1: Local Cases (LEFT) ===============
    n_positions = 3
    y_positions = np.array([0.22, 1.01, 1.8])
    
    legend_stats_local = []
    
    for i, country in enumerate(countries_to_plot):
        if country not in param_data['country'].values:
            print(f"Skipping {country} - not in data")
            continue
            
        country_row = param_data[param_data['country'] == country].iloc[0]
        local_cases_ww = country_row['WW_local_cases_samples_parsed']
        local_cases_ww = local_cases_ww[~np.isnan(local_cases_ww)]
        
        local_cases_icu = country_row['ICU_local_cases_samples_parsed']
        local_cases_icu = local_cases_icu[~np.isnan(local_cases_icu)]
        
        if len(local_cases_ww) == 0 or len(local_cases_icu) == 0:
            print(f"Skipping {country} - empty data")
            continue
        
        # Calculate statistics
        ww_mean = np.mean(local_cases_ww)
        ww_ci_low, ww_ci_high = np.percentile(local_cases_ww, [2.5, 97.5])
        icu_mean = np.mean(local_cases_icu)
        icu_ci_low, icu_ci_high = np.percentile(local_cases_icu, [2.5, 97.5])
        
        legend_stats_local.append({
            'country': country,
            'ww_mean': ww_mean,
            'ww_ci_low': ww_ci_low,
            'ww_ci_high': ww_ci_high,
            'icu_mean': icu_mean,
            'icu_ci_low': icu_ci_low,
            'icu_ci_high': icu_ci_high,
            'y_pos': y_positions[n_positions - 1 - i]
        })
        
        base_y = y_positions[n_positions - 1 - i]
        y_ww = base_y + 0.125
        y_icu = base_y - 0.125
        
        # WW scatter
        np.random.seed(42 + i)
        y_jitter_ww = np.random.normal(y_ww, 0.02, size=len(local_cases_ww))
        ax3.scatter(local_cases_ww, y_jitter_ww, alpha=0.7, s=9, color=color_ww, 
                   edgecolors='black', linewidths=0.3, zorder=2)
        
        # WW boxplot
        bp = ax3.boxplot([local_cases_ww], positions=[y_ww], vert=False, widths=0.10,
                         showfliers=False, patch_artist=True,
                         boxprops=dict(facecolor='white', alpha=0.5, linewidth=1.5, edgecolor='black'),
                         medianprops=dict(color='black', linewidth=2),
                         whiskerprops=dict(linewidth=1.5, color='black'),
                         capprops=dict(linewidth=1.5, color='black'),
                         zorder=3)
        
        # WW KDE
        if len(local_cases_ww) > 10:
            kde = stats.gaussian_kde(local_cases_ww, bw_method=0.3)
            x_range = np.linspace(max(0, local_cases_ww.min() - 5), local_cases_ww.max() + 5, 200)
            kde_values = kde(x_range)
            kde_scaled = kde_values / kde_values.max() * 0.22
            ax3.plot(x_range, kde_scaled + y_ww + 0.03, color=color_ww, linewidth=1, zorder=4)
        
        # ICU scatter
        np.random.seed(52 + i)
        y_jitter_icu = np.random.normal(y_icu, 0.02, size=len(local_cases_icu))
        ax3.scatter(local_cases_icu, y_jitter_icu, alpha=0.7, s=9, color=color_icu, 
                   edgecolors='black', linewidths=0.3, zorder=2)
        
        # ICU boxplot
        bp = ax3.boxplot([local_cases_icu], positions=[y_icu], vert=False, widths=0.10,
                         showfliers=False, patch_artist=True,
                         boxprops=dict(facecolor='white', alpha=0.5, linewidth=1.5, edgecolor='black'),
                         medianprops=dict(color='black', linewidth=2),
                         whiskerprops=dict(linewidth=1.5, color='black'),
                         capprops=dict(linewidth=1.5, color='black'),
                         zorder=3)
        
        # ICU KDE
        if len(local_cases_icu) > 10:
            kde = stats.gaussian_kde(local_cases_icu, bw_method=0.3)
            x_range = np.linspace(max(0, local_cases_icu.min() - 5), local_cases_icu.max() + 5, 200)
            kde_values = kde(x_range)
            kde_scaled = kde_values / kde_values.max() * 0.22
            ax3.plot(x_range, kde_scaled + y_icu + 0.03, color=color_icu, linewidth=1.5, zorder=4)
    
    # Add legend boxes for local cases (right side)
    xlim = ax3.get_xlim()
    if R0_val == 3.0:
        x_legend_start = xlim[1] * 0.775
    elif R0_val == 2.5 or (R0_val == 2.0 and gen_time_val == 4.0):
        x_legend_start = xlim[1] * 0.79
    elif R0_val == 2.0:
        x_legend_start = xlim[1] * 0.8
    else:
        x_legend_start = xlim[1] * 0.81
    box_width = (xlim[1] - xlim[0]) * 0.025
    text_offset = (xlim[1] - xlim[0]) * 0.035
    
    for stats_dict in legend_stats_local:
        y_pos = stats_dict['y_pos']
        
        # Determine which position this is (top, middle, bottom)
        if y_pos == y_positions[0]:  # Bottom position
            extra_shift = 0.25  # Double shift for bottom
        elif y_pos == y_positions[1]:  # Middle position
            extra_shift = 0.125  # Single shift for middle
        else:  # Top position
            extra_shift = 0.03  # No shift for top
        
        ww_y_offset = 0.15 + extra_shift
        icu_y_offset = 0.03 + extra_shift
        box_height = 0.065
        extra = (y_pos - 1.55)/14 + 0.08
        
        # WW legend
        ax3.add_patch(Rectangle((x_legend_start, y_pos + ww_y_offset + extra), box_width, box_height, 
                                fc=color_ww, ec='black', linewidth=1.5, zorder=10))
        ax3.text(x_legend_start + text_offset, y_pos + ww_y_offset + box_height/2 + extra,
                f"Mean: {stats_dict['ww_mean']:.1f}",
                fontsize=16, va='center', ha='left', zorder=10)
        
        # ICU legend
        ax3.add_patch(Rectangle((x_legend_start, y_pos + icu_y_offset + extra), box_width, box_height,
                                fc=color_icu, ec='black', linewidth=1.5, zorder=10))
        ax3.text(x_legend_start + text_offset, y_pos + icu_y_offset + box_height/2 + extra,
                f"Mean: {stats_dict['icu_mean']:.1f}",
                fontsize=16, va='center', ha='left', zorder=10)
    
    ax3.set_yticklabels([])
    ax3.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.8)
    ax3.set_ylim([0.0, 2.2])
    ax3.tick_params(axis='both', labelsize=20)
    
    # =============== PLOT 2: Detection Times (RIGHT) ===============
    legend_stats_detection = []
    
    for i, country in enumerate(countries_to_plot):
        if country not in param_data['country'].values:
            continue
            
        country_row = param_data[param_data['country'] == country].iloc[0]
        detection_times_ww = country_row['WW_detection_times_parsed']
        detection_times_ww = detection_times_ww[~np.isnan(detection_times_ww)]
        
        detection_times_icu = country_row['ICU_detection_times_parsed']
        detection_times_icu = detection_times_icu[~np.isnan(detection_times_icu)]
        
        if len(detection_times_ww) == 0 or len(detection_times_icu) == 0:
            continue
        
        # Calculate statistics
        ww_mean = np.mean(detection_times_ww)
        ww_ci_low, ww_ci_high = np.percentile(detection_times_ww, [2.5, 97.5])
        icu_mean = np.mean(detection_times_icu)
        icu_ci_low, icu_ci_high = np.percentile(detection_times_icu, [2.5, 97.5])
        
        legend_stats_detection.append({
            'country': country,
            'ww_mean': ww_mean,
            'ww_ci_low': ww_ci_low,
            'ww_ci_high': ww_ci_high,
            'icu_mean': icu_mean,
            'icu_ci_low': icu_ci_low,
            'icu_ci_high': icu_ci_high,
            'y_pos': y_positions[n_positions - 1 - i]
        })
        
        base_y = y_positions[n_positions - 1 - i]
        y_ww = base_y + 0.125
        y_icu = base_y - 0.125
        
        # WW scatter
        np.random.seed(62 + i)
        y_jitter_ww = np.random.normal(y_ww, 0.02, size=len(detection_times_ww))
        x_jitter_ww = np.random.normal(detection_times_ww, 0.12, size=len(detection_times_ww))
        ax4.scatter(x_jitter_ww, y_jitter_ww, alpha=0.7, s=9, color=color_ww,
                   edgecolors='black', linewidths=0.3, zorder=2)
        
        # WW boxplot
        bp = ax4.boxplot([detection_times_ww], positions=[y_ww], vert=False, widths=0.10,
                         showfliers=False, patch_artist=True,
                         boxprops=dict(facecolor='white', alpha=0.5, linewidth=1.5, edgecolor='black'),
                         medianprops=dict(color='black', linewidth=2),
                         whiskerprops=dict(linewidth=1.5, color='black'),
                         capprops=dict(linewidth=1.5, color='black'),
                         zorder=3)
        
        # WW KDE
        if len(detection_times_ww) > 10:
            kde = stats.gaussian_kde(detection_times_ww, bw_method=0.3)
            x_range = np.linspace(max(0, detection_times_ww.min() - 5), detection_times_ww.max() + 5, 200)
            kde_values = kde(x_range)
            kde_scaled = kde_values / kde_values.max() * 0.22
            ax4.plot(x_range, kde_scaled + y_ww + 0.03, color=color_ww, linewidth=1, zorder=4)
        
        # ICU scatter
        np.random.seed(72 + i)
        y_jitter_icu = np.random.normal(y_icu, 0.02, size=len(detection_times_icu))
        ax4.scatter(detection_times_icu, y_jitter_icu, alpha=0.7, s=9, color=color_icu,
                   edgecolors='black', linewidths=0.3, zorder=2)
        
        # ICU boxplot
        bp = ax4.boxplot([detection_times_icu], positions=[y_icu], vert=False, widths=0.10,
                         showfliers=False, patch_artist=True,
                         boxprops=dict(facecolor='white', alpha=0.5, linewidth=1.5, edgecolor='black'),
                         medianprops=dict(color='black', linewidth=2),
                         whiskerprops=dict(linewidth=1.5, color='black'),
                         capprops=dict(linewidth=1.5, color='black'),
                         zorder=3)
        
        # ICU KDE
        if len(detection_times_icu) > 10:
            kde = stats.gaussian_kde(detection_times_icu, bw_method=0.3)
            x_range = np.linspace(max(0, detection_times_icu.min() - 5), detection_times_icu.max() + 5, 200)
            kde_values = kde(x_range)
            kde_scaled = kde_values / kde_values.max() * 0.22
            ax4.plot(x_range, kde_scaled + y_icu + 0.03, color=color_icu, linewidth=1.5, zorder=4)
    
    # Add legend boxes for detection times (LEFT side)
    xlim4 = ax4.get_xlim()
    x_legend_start4 = xlim4[0] + (xlim4[1] - xlim4[0]) * 0.02  # Start near left edge
    box_width4 = (xlim4[1] - xlim4[0]) * 0.025
    text_offset4 = (xlim4[1] - xlim4[0]) * 0.035
    
    for stats_dict in legend_stats_detection:
        y_pos = stats_dict['y_pos']
        
        # Determine which position this is (top, middle, bottom)
        if y_pos == y_positions[0]:  # Bottom position
            extra_shift = 0.25  # Double shift for bottom
        elif y_pos == y_positions[1]:  # Middle position
            extra_shift = 0.125  # Single shift for middle
        else:  # Top position
            extra_shift = 0.03  # No shift for top
        
        ww_y_offset = 0.15 + extra_shift
        icu_y_offset = 0.03 + extra_shift
        box_height = 0.065
        extra = (y_pos - 1.55)/14 + 0.08
        
        # WW legend
        ax4.add_patch(Rectangle((x_legend_start4, y_pos + ww_y_offset + extra), box_width4, box_height,
                                fc=color_ww, ec='black', linewidth=1.5, zorder=10))
        ax4.text(x_legend_start4 + text_offset4, y_pos + ww_y_offset + box_height/2 + extra,
                f"Mean: {stats_dict['ww_mean']:.1f}",
                fontsize=16, va='center', ha='left', zorder=10)
        
        # ICU legend
        ax4.add_patch(Rectangle((x_legend_start4, y_pos + icu_y_offset + extra), box_width4, box_height,
                                fc=color_icu, ec='black', linewidth=1.5, zorder=10))
        ax4.text(x_legend_start4 + text_offset4, y_pos + icu_y_offset + box_height/2 + extra,
                f"Mean: {stats_dict['icu_mean']:.1f}",
                fontsize=16, va='center', ha='left', zorder=10)
    
    ax4.set_yticklabels([])
    ax4.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.8, zorder=1)
    ax4.set_ylim([0.0, 2.2])
    ax4.tick_params(axis='both', labelsize=20)
    
    plt.tight_layout()
    
    # Save figure
    output_filename = f'detection_comparison_R0_{R0_val}_gentime_{gen_time_val}.pdf'
    plt.savefig(output_filename, bbox_inches='tight')
    plt.show()
    print(f"Saved: {output_filename}")
    
    # Print summary statistics
    print(f"\nSummary for R0={R0_val}, gen_time={gen_time_val}:")
    print("\n--- LOCAL CASES ---")
    for stats_dict in legend_stats_local:
        print(f"{stats_dict['country']}:")
        print(f"  WW: {stats_dict['ww_mean']:.2f} ({stats_dict['ww_ci_low']:.2f}-{stats_dict['ww_ci_high']:.2f})")
        print(f"  ICU: {stats_dict['icu_mean']:.2f} ({stats_dict['icu_ci_low']:.2f}-{stats_dict['icu_ci_high']:.2f})")
    
    print("\n--- DETECTION TIMES ---")
    for stats_dict in legend_stats_detection:
        print(f"{stats_dict['country']}:")
        print(f"  WW: {stats_dict['ww_mean']:.2f} ({stats_dict['ww_ci_low']:.2f}-{stats_dict['ww_ci_high']:.2f})")
        print(f"  ICU: {stats_dict['icu_mean']:.2f} ({stats_dict['icu_ci_low']:.2f}-{stats_dict['icu_ci_high']:.2f})")
    
    plt.close()

print("\n✓ All plots generated!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import ast
import re
from matplotlib.patches import Rectangle

# Helper function to parse arrays safely
def safe_parse_array(x):
    if pd.isna(x) or x is None:
        return np.array([])
    if isinstance(x, str):
        if x.strip() == 'Float64[]' or x.strip() == '[]':
            return np.array([])
        x = re.sub(r'Float64\[', '[', x)
        try:
            return np.array(ast.literal_eval(x))
        except:
            return np.array([])
    return x

# Load the dataset
df = pd.read_csv('pgfgleam/all_results/local/aww_t3_comparison_results.csv')

print("Dataset columns:", df.columns.tolist())
print("Dataset shape:", df.shape)
print("Unique countries:", df['country'].unique())
print("Unique R0 values:", df['R0'].unique())
print("Unique gen_time values:", df['gen_time'].unique())

# Use the mean columns directly (already computed)
df_valid = df[
    ~df['AWW_ALL_016_10pct_mean_detection_time'].isna() &
    ~df['AWW_TOP3_016_10pct_mean_detection_time'].isna() &
    ~df['AWW_ALL_016_10pct_mean_local_cases'].isna() &
    ~df['AWW_TOP3_016_10pct_mean_local_cases'].isna()
].copy()

print(f"\nAfter filtering: {len(df_valid)} rows with valid data")

# Get unique parameter combinations
param_combinations = df_valid[['R0', 'gen_time']].drop_duplicates().sort_values(['R0', 'gen_time'])
print(f"\nParameter combinations to plot: {len(param_combinations)}")
print(param_combinations)

# Define colors (turquoise for ALL, orange for TOP3)
color_all = '#4FB3BE'  # Turquoise for ALL countries
color_top3 = '#E87D47'  # Orange for TOP3

# Create a plot for each (R0, gen_time) combination
for idx, (_, param_row) in enumerate(param_combinations.iterrows()):
    R0_val = param_row['R0']
    gen_time_val = param_row['gen_time']
    
    print(f"\n{'='*60}")
    print(f"Plotting R0={R0_val}, gen_time={gen_time_val}")
    print(f"{'='*60}")
    
    # Filter data for this parameter combination (all countries)
    param_data = df_valid[
        (df_valid['R0'] == R0_val) & 
        (df_valid['gen_time'] == gen_time_val)
    ].copy()
    
    if len(param_data) == 0:
        print(f"No data for R0={R0_val}, gen_time={gen_time_val}")
        continue
    
    print(f"Number of countries: {len(param_data)}")
    print(f"Countries: {param_data['country'].tolist()}")
    
    # Extract mean values across all countries
    detection_times_all = param_data['AWW_ALL_016_10pct_mean_detection_time'].values
    detection_times_top3 = param_data['AWW_TOP3_016_10pct_mean_detection_time'].values
    local_cases_all = param_data['AWW_ALL_016_10pct_mean_local_cases'].values
    local_cases_top3 = param_data['AWW_TOP3_016_10pct_mean_local_cases'].values
    
    # Create figure with 2 panels
    fig, (ax_detection, ax_cases) = plt.subplots(1, 2, figsize=(20, 7))
    
    # =============== PLOT 1: Detection Times (LEFT) ===============
    y_pos_all = 1.2
    y_pos_top3 = 0.8
    
    # Calculate overall statistics
    all_det_mean = np.mean(detection_times_all)
    all_det_ci_low, all_det_ci_high = np.percentile(detection_times_all, [2.5, 97.5])
    top3_det_mean = np.mean(detection_times_top3)
    top3_det_ci_low, top3_det_ci_high = np.percentile(detection_times_top3, [2.5, 97.5])
    
    # ALL scatter
    np.random.seed(42)
    y_jitter_all = np.random.normal(y_pos_all, 0.02, size=len(detection_times_all))
    x_jitter_all = np.random.normal(detection_times_all, 0.12, size=len(detection_times_all))
    ax_detection.scatter(x_jitter_all, y_jitter_all, alpha=0.7, s=20, color=color_all,
                        edgecolors='black', linewidths=0.5, zorder=2, label='ALL countries')
    
    # ALL boxplot
    bp = ax_detection.boxplot([detection_times_all], positions=[y_pos_all], vert=False, widths=0.15,
                              showfliers=False, patch_artist=True,
                              boxprops=dict(facecolor='white', alpha=0.5, linewidth=2, edgecolor='black'),
                              medianprops=dict(color='black', linewidth=2.5),
                              whiskerprops=dict(linewidth=2, color='black'),
                              capprops=dict(linewidth=2, color='black'),
                              zorder=3)
    
    # ALL KDE
    if len(detection_times_all) > 5:
        kde = stats.gaussian_kde(detection_times_all, bw_method=0.3)
        x_range = np.linspace(max(0, detection_times_all.min() - 5), detection_times_all.max() + 5, 200)
        kde_values = kde(x_range)
        kde_scaled = kde_values / kde_values.max() * 0.3
        ax_detection.plot(x_range, kde_scaled + y_pos_all + 0.05, color=color_all, linewidth=2, zorder=4)
    
    # TOP3 scatter
    np.random.seed(52)
    y_jitter_top3 = np.random.normal(y_pos_top3, 0.02, size=len(detection_times_top3))
    x_jitter_top3 = np.random.normal(detection_times_top3, 0.12, size=len(detection_times_top3))
    ax_detection.scatter(x_jitter_top3, y_jitter_top3, alpha=0.7, s=20, color=color_top3,
                        edgecolors='black', linewidths=0.5, zorder=2, label='TOP-3 countries')
    
    # TOP3 boxplot
    bp = ax_detection.boxplot([detection_times_top3], positions=[y_pos_top3], vert=False, widths=0.15,
                              showfliers=False, patch_artist=True,
                              boxprops=dict(facecolor='white', alpha=0.5, linewidth=2, edgecolor='black'),
                              medianprops=dict(color='black', linewidth=2.5),
                              whiskerprops=dict(linewidth=2, color='black'),
                              capprops=dict(linewidth=2, color='black'),
                              zorder=3)
    
    # TOP3 KDE
    if len(detection_times_top3) > 5:
        kde = stats.gaussian_kde(detection_times_top3, bw_method=0.3)
        x_range = np.linspace(max(0, detection_times_top3.min() - 5), detection_times_top3.max() + 5, 200)
        kde_values = kde(x_range)
        kde_scaled = kde_values / kde_values.max() * 0.3
        ax_detection.plot(x_range, kde_scaled + y_pos_top3 + 0.05, color=color_top3, linewidth=2, zorder=4)
    
    # Add legend boxes for detection times
    xlim_det = ax_detection.get_xlim()
    x_legend_start = xlim_det[0] + (xlim_det[1] - xlim_det[0]) * 0.02
    box_width = (xlim_det[1] - xlim_det[0]) * 0.035
    text_offset = (xlim_det[1] - xlim_det[0]) * 0.045
    box_height = 0.1
    
    # ALL legend
    ax_detection.add_patch(Rectangle((x_legend_start, y_pos_all + 0.05), box_width, box_height,
                                     fc=color_all, ec='black', linewidth=2, zorder=10))
    ax_detection.text(x_legend_start + text_offset, y_pos_all + 0.1,
                     f"Mean: {all_det_mean:.1f}d\n95% CI: [{all_det_ci_low:.1f}, {all_det_ci_high:.1f}]",
                     fontsize=18, va='center', ha='left', zorder=10)
    
    # TOP3 legend
    ax_detection.add_patch(Rectangle((x_legend_start, y_pos_top3 + 0.05), box_width, box_height,
                                     fc=color_top3, ec='black', linewidth=2, zorder=10))
    ax_detection.text(x_legend_start + text_offset, y_pos_top3 + 0.1,
                     f"Mean: {top3_det_mean:.1f}d\n95% CI: [{top3_det_ci_low:.1f}, {top3_det_ci_high:.1f}]",
                     fontsize=18, va='center', ha='left', zorder=10)
    
    ax_detection.set_xlabel('Detection Time (days)', fontsize=22)
    ax_detection.set_yticks([])
    ax_detection.grid(axis='x', alpha=0.3, linestyle='--', linewidth=1)
    ax_detection.set_ylim([0.4, 1.6])
    ax_detection.tick_params(axis='x', labelsize=20)
    ax_detection.set_title(f'Detection Time\n$R_0$={R0_val}, Gen Time={gen_time_val}d', fontsize=22, pad=15)
    
    # =============== PLOT 2: Local Cases (RIGHT) ===============
    # Calculate overall statistics
    all_cases_mean = np.mean(local_cases_all)
    all_cases_ci_low, all_cases_ci_high = np.percentile(local_cases_all, [2.5, 97.5])
    top3_cases_mean = np.mean(local_cases_top3)
    top3_cases_ci_low, top3_cases_ci_high = np.percentile(local_cases_top3, [2.5, 97.5])
    
    # ALL scatter
    np.random.seed(62)
    y_jitter_all = np.random.normal(y_pos_all, 0.02, size=len(local_cases_all))
    ax_cases.scatter(local_cases_all, y_jitter_all, alpha=0.7, s=20, color=color_all,
                    edgecolors='black', linewidths=0.5, zorder=2)
    
    # ALL boxplot
    bp = ax_cases.boxplot([local_cases_all], positions=[y_pos_all], vert=False, widths=0.15,
                          showfliers=False, patch_artist=True,
                          boxprops=dict(facecolor='white', alpha=0.5, linewidth=2, edgecolor='black'),
                          medianprops=dict(color='black', linewidth=2.5),
                          whiskerprops=dict(linewidth=2, color='black'),
                          capprops=dict(linewidth=2, color='black'),
                          zorder=3)
    
    # ALL KDE
    if len(local_cases_all) > 5:
        kde = stats.gaussian_kde(local_cases_all, bw_method=0.3)
        x_range = np.linspace(max(0, local_cases_all.min() - 5), local_cases_all.max() + 5, 200)
        kde_values = kde(x_range)
        kde_scaled = kde_values / kde_values.max() * 0.3
        ax_cases.plot(x_range, kde_scaled + y_pos_all + 0.05, color=color_all, linewidth=2, zorder=4)
    
    # TOP3 scatter
    np.random.seed(72)
    y_jitter_top3 = np.random.normal(y_pos_top3, 0.02, size=len(local_cases_top3))
    ax_cases.scatter(local_cases_top3, y_jitter_top3, alpha=0.7, s=20, color=color_top3,
                    edgecolors='black', linewidths=0.5, zorder=2)
    
    # TOP3 boxplot
    bp = ax_cases.boxplot([local_cases_top3], positions=[y_pos_top3], vert=False, widths=0.15,
                          showfliers=False, patch_artist=True,
                          boxprops=dict(facecolor='white', alpha=0.5, linewidth=2, edgecolor='black'),
                          medianprops=dict(color='black', linewidth=2.5),
                          whiskerprops=dict(linewidth=2, color='black'),
                          capprops=dict(linewidth=2, color='black'),
                          zorder=3)
    
    # TOP3 KDE
    if len(local_cases_top3) > 5:
        kde = stats.gaussian_kde(local_cases_top3, bw_method=0.3)
        x_range = np.linspace(max(0, local_cases_top3.min() - 5), local_cases_top3.max() + 5, 200)
        kde_values = kde(x_range)
        kde_scaled = kde_values / kde_values.max() * 0.3
        ax_cases.plot(x_range, kde_scaled + y_pos_top3 + 0.05, color=color_top3, linewidth=2, zorder=4)
    
    # Add legend boxes for local cases
    xlim_cases = ax_cases.get_xlim()
    x_legend_start = xlim_cases[1] * 0.75
    box_width = (xlim_cases[1] - xlim_cases[0]) * 0.035
    text_offset = (xlim_cases[1] - xlim_cases[0]) * 0.045
    
    # ALL legend
    ax_cases.add_patch(Rectangle((x_legend_start, y_pos_all + 0.05), box_width, box_height,
                                 fc=color_all, ec='black', linewidth=2, zorder=10))
    ax_cases.text(x_legend_start + text_offset, y_pos_all + 0.1,
                 f"Mean: {all_cases_mean:.1f}\n95% CI: [{all_cases_ci_low:.1f}, {all_cases_ci_high:.1f}]",
                 fontsize=18, va='center', ha='left', zorder=10)
    
    # TOP3 legend
    ax_cases.add_patch(Rectangle((x_legend_start, y_pos_top3 + 0.05), box_width, box_height,
                                 fc=color_top3, ec='black', linewidth=2, zorder=10))
    ax_cases.text(x_legend_start + text_offset, y_pos_top3 + 0.1,
                 f"Mean: {top3_cases_mean:.1f}\n95% CI: [{top3_cases_ci_low:.1f}, {top3_cases_ci_high:.1f}]",
                 fontsize=18, va='center', ha='left', zorder=10)
    
    ax_cases.set_xlabel('Local Cases at Detection', fontsize=22)
    ax_cases.set_yticks([])
    ax_cases.grid(axis='x', alpha=0.3, linestyle='--', linewidth=1)
    ax_cases.set_ylim([0.4, 1.6])
    ax_cases.tick_params(axis='x', labelsize=20)
    ax_cases.set_title(f'Local Cases at Detection\n$R_0$={R0_val}, Gen Time={gen_time_val}d', fontsize=22, pad=15)
    
    plt.tight_layout()
    
    # Save figure
    output_filename = f'aww_t3_means_comparison_R0_{R0_val}_gentime_{gen_time_val}.pdf'
    # plt.savefig(output_filename, bbox_inches='tight', dpi=300)
    plt.show()
    print(f"Saved: {output_filename}")
    
    # Print summary statistics
    print(f"\nSummary for R0={R0_val}, gen_time={gen_time_val}:")
    print(f"Number of countries: {len(param_data)}")
    print("\n--- DETECTION TIMES (across all countries) ---")
    print(f"  ALL: {all_det_mean:.2f} ({all_det_ci_low:.2f}-{all_det_ci_high:.2f}) days")
    print(f"  TOP3: {top3_det_mean:.2f} ({top3_det_ci_low:.2f}-{top3_det_ci_high:.2f}) days")
    print(f"  Difference (TOP3 - ALL): {top3_det_mean - all_det_mean:.2f} days")
    
    print("\n--- LOCAL CASES (across all countries) ---")
    print(f"  ALL: {all_cases_mean:.2f} ({all_cases_ci_low:.2f}-{all_cases_ci_high:.2f})")
    print(f"  TOP3: {top3_cases_mean:.2f} ({top3_cases_ci_low:.2f}-{top3_cases_ci_high:.2f})")
    print(f"  Ratio (TOP3 / ALL): {top3_cases_mean / all_cases_mean:.2f}")
    
    plt.close()

print("\n✓ All plots generated!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import ast
import re
from matplotlib.patches import Rectangle

# Helper function to parse arrays safely
def safe_parse_array(x):
    if pd.isna(x) or x is None:
        return np.array([])
    if isinstance(x, str):
        if x.strip() == 'Float64[]' or x.strip() == '[]':
            return np.array([])
        x = re.sub(r'Float64\[', '[', x)
        try:
            return np.array(ast.literal_eval(x))
        except:
            return np.array([])
    return x

# Load the dataset
df = pd.read_csv('pgfgleam/all_results/local/full_results.csv')

print("Dataset columns:", df.columns.tolist())
print("Dataset shape:", df.shape)
print("Unique countries:", df['country'].unique())
print("Unique R0 values:", df['R0'].unique())
print("Unique gen_time values:", df['gen_time'].unique())

# Filter for R0=2.0 and gen_time=4.0
df_filtered = df[
    (df['R0'] == 2.0) & 
    (df['gen_time'] == 4.0)
].copy()

print(f"\nFiltered to R0=2.0, gen_time=4.0: {len(df_filtered)} countries")
print(f"Countries: {df_filtered['country'].tolist()}")

if len(df_filtered) == 0:
    print("No data found for R0=2.0, gen_time=4.0!")
    exit()

# Extract mean values (already computed as means across simulations for each country)
aww_detection_times = df_filtered['WW_016_25pct_mean_detection_time'].values
icu_detection_times = df_filtered['ICU_mean_detection_time'].values
aww_local_cases = df_filtered['WW_016_25pct_mean_local_cases'].values
icu_local_cases = df_filtered['ICU_mean_local_cases'].values

# Define colors (turquoise for AWW, orange for ICU)
color_aww = '#4FB3BE'  # Turquoise
color_icu = '#E87D47'  # Orange

# Create figure with 2 panels
fig, (ax_detection, ax_cases) = plt.subplots(1, 2, figsize=(20, 7))

# =============== PLOT 1: Detection Times (LEFT) ===============
y_pos_aww = 1.2
y_pos_icu = 0.8

# Calculate overall statistics
aww_det_mean = np.mean(aww_detection_times)
aww_det_ci_low, aww_det_ci_high = np.percentile(aww_detection_times, [2.5, 97.5])
icu_det_mean = np.mean(icu_detection_times)
icu_det_ci_low, icu_det_ci_high = np.percentile(icu_detection_times, [2.5, 97.5])

# AWW scatter
np.random.seed(42)
y_jitter_aww = np.random.normal(y_pos_aww, 0.02, size=len(aww_detection_times))
x_jitter_aww = np.random.normal(aww_detection_times, 0.12, size=len(aww_detection_times))
ax_detection.scatter(x_jitter_aww, y_jitter_aww, alpha=0.7, s=20, color=color_aww,
                    edgecolors='black', linewidths=0.5, zorder=2, label='AWW')

# AWW boxplot
bp = ax_detection.boxplot([aww_detection_times], positions=[y_pos_aww], vert=False, widths=0.15,
                          showfliers=False, patch_artist=True,
                          boxprops=dict(facecolor='white', alpha=0.5, linewidth=2, edgecolor='black'),
                          medianprops=dict(color='black', linewidth=2.5),
                          whiskerprops=dict(linewidth=2, color='black'),
                          capprops=dict(linewidth=2, color='black'),
                          zorder=3)

# AWW KDE
if len(aww_detection_times) > 5:
    kde = stats.gaussian_kde(aww_detection_times, bw_method=0.3)
    x_range = np.linspace(max(0, aww_detection_times.min() - 5), aww_detection_times.max() + 5, 200)
    kde_values = kde(x_range)
    kde_scaled = kde_values / kde_values.max() * 0.3
    ax_detection.plot(x_range, kde_scaled + y_pos_aww + 0.05, color=color_aww, linewidth=2, zorder=4)

# ICU scatter
np.random.seed(52)
y_jitter_icu = np.random.normal(y_pos_icu, 0.02, size=len(icu_detection_times))
x_jitter_icu = np.random.normal(icu_detection_times, 0.12, size=len(icu_detection_times))
ax_detection.scatter(x_jitter_icu, y_jitter_icu, alpha=0.7, s=20, color=color_icu,
                    edgecolors='black', linewidths=0.5, zorder=2, label='ICU')

# ICU boxplot
bp = ax_detection.boxplot([icu_detection_times], positions=[y_pos_icu], vert=False, widths=0.15,
                          showfliers=False, patch_artist=True,
                          boxprops=dict(facecolor='white', alpha=0.5, linewidth=2, edgecolor='black'),
                          medianprops=dict(color='black', linewidth=2.5),
                          whiskerprops=dict(linewidth=2, color='black'),
                          capprops=dict(linewidth=2, color='black'),
                          zorder=3)

# ICU KDE
if len(icu_detection_times) > 5:
    kde = stats.gaussian_kde(icu_detection_times, bw_method=0.3)
    x_range = np.linspace(max(0, icu_detection_times.min() - 5), icu_detection_times.max() + 5, 200)
    kde_values = kde(x_range)
    kde_scaled = kde_values / kde_values.max() * 0.3
    ax_detection.plot(x_range, kde_scaled + y_pos_icu + 0.05, color=color_icu, linewidth=2, zorder=4)

# Add legend boxes for detection times
xlim_det = ax_detection.get_xlim()
x_legend_start = xlim_det[0] + (xlim_det[1] - xlim_det[0]) * 0.02
box_width = (xlim_det[1] - xlim_det[0]) * 0.035
text_offset = (xlim_det[1] - xlim_det[0]) * 0.045
box_height = 0.1

# AWW legend
ax_detection.add_patch(Rectangle((x_legend_start, y_pos_aww + 0.05), box_width, box_height,
                                 fc=color_aww, ec='black', linewidth=2, zorder=10))
ax_detection.text(x_legend_start + text_offset, y_pos_aww + 0.1,
                 f"Mean: {aww_det_mean:.1f}d\n95% CI: [{aww_det_ci_low:.1f}, {aww_det_ci_high:.1f}]",
                 fontsize=18, va='center', ha='left', zorder=10)

# ICU legend
ax_detection.add_patch(Rectangle((x_legend_start, y_pos_icu + 0.05), box_width, box_height,
                                 fc=color_icu, ec='black', linewidth=2, zorder=10))
ax_detection.text(x_legend_start + text_offset, y_pos_icu + 0.1,
                 f"Mean: {icu_det_mean:.1f}d\n95% CI: [{icu_det_ci_low:.1f}, {icu_det_ci_high:.1f}]",
                 fontsize=18, va='center', ha='left', zorder=10)

ax_detection.set_xlabel('Detection Time (days)', fontsize=22)
ax_detection.set_yticks([])
ax_detection.grid(axis='x', alpha=0.3, linestyle='--', linewidth=1)
ax_detection.set_ylim([0.4, 1.6])
ax_detection.tick_params(axis='x', labelsize=20)
ax_detection.set_title(f'Detection Time\n$R_0$=2.0, Gen Time=4.0d', fontsize=22, pad=15)

# =============== PLOT 2: Local Cases (RIGHT) ===============
# Calculate overall statistics
aww_cases_mean = np.mean(aww_local_cases)
aww_cases_ci_low, aww_cases_ci_high = np.percentile(aww_local_cases, [2.5, 97.5])
icu_cases_mean = np.mean(icu_local_cases)
icu_cases_ci_low, icu_cases_ci_high = np.percentile(icu_local_cases, [2.5, 97.5])

# AWW scatter
np.random.seed(62)
y_jitter_aww = np.random.normal(y_pos_aww, 0.02, size=len(aww_local_cases))
ax_cases.scatter(aww_local_cases, y_jitter_aww, alpha=0.7, s=20, color=color_aww,
                edgecolors='black', linewidths=0.5, zorder=2)

# AWW boxplot
bp = ax_cases.boxplot([aww_local_cases], positions=[y_pos_aww], vert=False, widths=0.15,
                      showfliers=False, patch_artist=True,
                      boxprops=dict(facecolor='white', alpha=0.5, linewidth=2, edgecolor='black'),
                      medianprops=dict(color='black', linewidth=2.5),
                      whiskerprops=dict(linewidth=2, color='black'),
                      capprops=dict(linewidth=2, color='black'),
                      zorder=3)

# AWW KDE
if len(aww_local_cases) > 5:
    kde = stats.gaussian_kde(aww_local_cases, bw_method=0.3)
    x_range = np.linspace(max(0, aww_local_cases.min() - 5), aww_local_cases.max() + 5, 200)
    kde_values = kde(x_range)
    kde_scaled = kde_values / kde_values.max() * 0.3
    ax_cases.plot(x_range, kde_scaled + y_pos_aww + 0.05, color=color_aww, linewidth=2, zorder=4)

# ICU scatter
np.random.seed(72)
y_jitter_icu = np.random.normal(y_pos_icu, 0.02, size=len(icu_local_cases))
ax_cases.scatter(icu_local_cases, y_jitter_icu, alpha=0.7, s=20, color=color_icu,
                edgecolors='black', linewidths=0.5, zorder=2)

# ICU boxplot
bp = ax_cases.boxplot([icu_local_cases], positions=[y_pos_icu], vert=False, widths=0.15,
                      showfliers=False, patch_artist=True,
                      boxprops=dict(facecolor='white', alpha=0.5, linewidth=2, edgecolor='black'),
                      medianprops=dict(color='black', linewidth=2.5),
                      whiskerprops=dict(linewidth=2, color='black'),
                      capprops=dict(linewidth=2, color='black'),
                      zorder=3)

# ICU KDE
if len(icu_local_cases) > 5:
    kde = stats.gaussian_kde(icu_local_cases, bw_method=0.3)
    x_range = np.linspace(max(0, icu_local_cases.min() - 5), icu_local_cases.max() + 5, 200)
    kde_values = kde(x_range)
    kde_scaled = kde_values / kde_values.max() * 0.3
    ax_cases.plot(x_range, kde_scaled + y_pos_icu + 0.05, color=color_icu, linewidth=2, zorder=4)

# Add legend boxes for local cases
xlim_cases = ax_cases.get_xlim()
x_legend_start = xlim_cases[1] * 0.75
box_width = (xlim_cases[1] - xlim_cases[0]) * 0.035
text_offset = (xlim_cases[1] - xlim_cases[0]) * 0.045

# AWW legend
ax_cases.add_patch(Rectangle((x_legend_start, y_pos_aww + 0.05), box_width, box_height,
                             fc=color_aww, ec='black', linewidth=2, zorder=10))
ax_cases.text(x_legend_start + text_offset, y_pos_aww + 0.1,
             f"Mean: {aww_cases_mean:.1f}\n95% CI: [{aww_cases_ci_low:.1f}, {aww_cases_ci_high:.1f}]",
             fontsize=18, va='center', ha='left', zorder=10)

# ICU legend
ax_cases.add_patch(Rectangle((x_legend_start, y_pos_icu + 0.05), box_width, box_height,
                             fc=color_icu, ec='black', linewidth=2, zorder=10))
ax_cases.text(x_legend_start + text_offset, y_pos_icu + 0.1,
             f"Mean: {icu_cases_mean:.1f}\n95% CI: [{icu_cases_ci_low:.1f}, {icu_cases_ci_high:.1f}]",
             fontsize=18, va='center', ha='left', zorder=10)

ax_cases.set_xlabel('Local Cases at Detection', fontsize=22)
ax_cases.set_yticks([])
ax_cases.grid(axis='x', alpha=0.3, linestyle='--', linewidth=1)
ax_cases.set_ylim([0.4, 1.6])
ax_cases.tick_params(axis='x', labelsize=20)
ax_cases.set_title(f'Local Cases at Detection\n$R_0$=2.0, Gen Time=4.0d', fontsize=22, pad=15)

plt.tight_layout()

# Save figure
output_filename = 'aww_vs_icu_comparison_R0_2.0_gentime_4.0.pdf'
plt.savefig(output_filename, bbox_inches='tight', dpi=300)
plt.show()
print(f"\nSaved: {output_filename}")

# Calculate sample-by-sample differences and ratios
detection_time_diffs = icu_detection_times - aww_detection_times
local_cases_ratios = icu_local_cases / aww_local_cases

# Print summary statistics
print(f"\nSummary for R0=2.0, gen_time=4.0:")
print(f"Number of countries: {len(df_filtered)}")

print("\n--- DETECTION TIMES ---")
print(f"  AWW: {aww_det_mean:.2f} ({aww_det_ci_low:.2f}-{aww_det_ci_high:.2f}) days")
print(f"  ICU: {icu_det_mean:.2f} ({icu_det_ci_low:.2f}-{icu_det_ci_high:.2f}) days")
print(f"\n  Sample-by-sample differences (ICU - AWW):")
print(f"    Mean: {np.mean(detection_time_diffs):.2f} days")
print(f"    Range: [{np.min(detection_time_diffs):.2f}, {np.max(detection_time_diffs):.2f}] days")
print(f"    95% CI: [{np.percentile(detection_time_diffs, 2.5):.2f}, {np.percentile(detection_time_diffs, 97.5):.2f}] days")
print(f"    Median: {np.median(detection_time_diffs):.2f} days")

print("\n--- LOCAL CASES ---")
print(f"  AWW: {aww_cases_mean:.2f} ({aww_cases_ci_low:.2f}-{aww_cases_ci_high:.2f})")
print(f"  ICU: {icu_cases_mean:.2f} ({icu_cases_ci_low:.2f}-{icu_cases_ci_high:.2f})")
print(f"\n  Sample-by-sample ratios (ICU / AWW):")
print(f"    Mean: {np.mean(local_cases_ratios):.2f}x")
print(f"    Range: [{np.min(local_cases_ratios):.2f}x, {np.max(local_cases_ratios):.2f}x]")
print(f"    95% CI: [{np.percentile(local_cases_ratios, 2.5):.2f}x, {np.percentile(local_cases_ratios, 97.5):.2f}x]")
print(f"    Median: {np.median(local_cases_ratios):.2f}x")

print("\n✓ Plot generated!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import seaborn as sns

# Set style
sns.set_style("whitegrid")

# Load the simulated arrival and detection data
arrival_detection_data = pd.read_csv("pgfgleam/all_results/global/simulated_arrival_vs_detection_times.csv")

# Clean the arrival_detection_data
clean_data = arrival_detection_data.dropna(subset=['mean_arrival_time', 'mean_detection_time'])
clean_data = clean_data[(clean_data['mean_arrival_time'] > 0) & (clean_data['mean_detection_time'] > 0)]

print(f"Using data from {len(clean_data)} countries")

# Create flow data using arrival_detection_data
flow_data = []
uk_idx = source_countries.index('United Kingdom')

for _, row in clean_data.iterrows():
    country = row['country']
    
    # Skip UK
    if country == 'United Kingdom':
        continue
    
    try:
        country_idx = source_countries.index(country)
    except ValueError:
        print(f"Skipping {country} - not in source_countries list")
        continue
    
    # Calculate UK population proportion of travellers
    uk_pop_traveller_prop = normalized_flow_matrix[country_idx, uk_idx]
    
    flow_data.append({
        'country': country,
        'arrival_time': row['mean_arrival_time'],
        'detection_time': row['mean_detection_time'],
        'uk_pop_traveller_proportion': uk_pop_traveller_prop
    })

df_flow = pd.DataFrame(flow_data)

# Remove any infinite or NaN values, and zero/negative values (for log scale)
df_flow = df_flow[np.isfinite(df_flow['uk_pop_traveller_proportion'])]
df_flow = df_flow[np.isfinite(df_flow['arrival_time'])]
df_flow = df_flow[df_flow['uk_pop_traveller_proportion'] > 1e-8]

print(f"Clean data shape (travel prob plot): {len(df_flow)} countries")

fig2, ax2 = plt.subplots(figsize=(6, 5))

# Scatter plot - light green with black edges
scatter = ax2.scatter(
    df_flow['arrival_time'].values,
    df_flow['uk_pop_traveller_proportion'].values,
    c='lightgreen',
    s=100,
    alpha=0.7,
    edgecolors='black',
    linewidth=1
)

# Use log scale for y-axis
ax2.set_yscale('log')

# Remove labels and title (matching minimal style)
ax2.set_xlabel('')
ax2.set_ylabel('')
ax2.set_title('')

# Grid and formatting
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.tick_params(axis='both', which='major', labelsize=12)

plt.tight_layout()
plt.savefig('uk_travel_prob_vs_arrival_time.pdf', dpi=300, bbox_inches='tight')
print("Saved: uk_travel_prob_vs_arrival_time.pdf")

# Calculate log-linear regression for travel probability plot
log_y_data = np.log(df_flow['uk_pop_traveller_proportion'].values)
x_data = df_flow['arrival_time'].values
slope_travel, intercept_travel, r_travel, p_travel, std_err = stats.linregress(x_data, log_y_data)

# Print countries far away from the linear regression line
predicted_log_y = intercept_travel + slope_travel * x_data
residuals = log_y_data - predicted_log_y
std_residuals = np.std(residuals)
outlier_threshold = 2 * std_residuals
outliers = df_flow[np.abs(residuals) > outlier_threshold]

if len(outliers) > 0:
    print("\nCountries that are outliers in the UK travel probability plot:")
    for _, row in outliers.iterrows():
        print(f"  {row['country']}: Arrival Time = {row['arrival_time']:.1f}, UK Travel Proportion = {row['uk_pop_traveller_proportion']:.6f}")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import griddata
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

plt.rcParams.update({'font.size': 18})

# Load data
df = pd.read_csv("pgfgleam/all_results/local/aww_t3_comparison_results.csv")

# Filter valid data
df_filtered = df[
    ~df['AWW_ALL_016_10pct_mean_detection_time'].isna() &
    ~df['AWW_TOP3_016_10pct_mean_detection_time'].isna() &
    ~df['AWW_ALL_016_10pct_mean_local_cases'].isna() &
    ~df['AWW_TOP3_016_10pct_mean_local_cases'].isna()
].copy()

print(f"Total rows with valid data: {len(df_filtered)}")
print(f"Unique R0 values: {sorted(df_filtered['R0'].unique())}")
print(f"Unique gen_time values: {sorted(df_filtered['gen_time'].unique())}")
print(f"Number of countries: {df_filtered['country'].nunique()}")

# ============================================================================
# AGGREGATE OVER COUNTRIES - Get mean of means for each (R0, gen_time) combo
# ============================================================================

configs_016 = [
    ('016_10pct', '10% sampling (eff=1.6%)'),
    ('016_25pct', '25% sampling (eff=4.0%)'),
    ('016_50pct', '50% sampling (eff=8.0%)'),
    ('016_100pct', '100% sampling (eff=16.0%)')
]

# Group by R0 and gen_time, then take mean across all countries
aggregated_data = df_filtered.groupby(['R0', 'gen_time']).agg({
    'AWW_ALL_016_10pct_mean_detection_time': 'mean',
    'AWW_TOP3_016_10pct_mean_detection_time': 'mean',
    'AWW_ALL_016_10pct_mean_local_cases': 'mean',
    'AWW_TOP3_016_10pct_mean_local_cases': 'mean',
    'AWW_ALL_016_25pct_mean_detection_time': 'mean',
    'AWW_TOP3_016_25pct_mean_detection_time': 'mean',
    'AWW_ALL_016_25pct_mean_local_cases': 'mean',
    'AWW_TOP3_016_25pct_mean_local_cases': 'mean',
    'AWW_ALL_016_50pct_mean_detection_time': 'mean',
    'AWW_TOP3_016_50pct_mean_detection_time': 'mean',
    'AWW_ALL_016_50pct_mean_local_cases': 'mean',
    'AWW_TOP3_016_50pct_mean_local_cases': 'mean',
    'AWW_ALL_016_100pct_mean_detection_time': 'mean',
    'AWW_TOP3_016_100pct_mean_detection_time': 'mean',
    'AWW_ALL_016_100pct_mean_local_cases': 'mean',
    'AWW_TOP3_016_100pct_mean_local_cases': 'mean',
    'country': 'count'  # Count number of countries
}).reset_index()

# Rename country column to n_countries
aggregated_data.rename(columns={'country': 'n_countries'}, inplace=True)

print(f"\nAggregated data points: {len(aggregated_data)}")
print(aggregated_data[['R0', 'gen_time', 'n_countries']].head(10))

# Prepare data for plotting
R0_data = aggregated_data['R0'].values
gen_time_data = aggregated_data['gen_time'].values

# Create grid
R0_grid = np.linspace(1.5, 3.0, 200)
gen_time_grid = np.linspace(4.0, 10.0, 200)
R0_mesh, gen_time_mesh = np.meshgrid(R0_grid, gen_time_grid)

cmap = 'viridis'

# ============================================================================
# FIGURE: BASE P_DET = 16% - TOP-3 vs ALL COMPARISON (AGGREGATED MEANS)
# ============================================================================
fig = plt.figure(figsize=(25, 11))
gs = fig.add_gridspec(2, 4, hspace=0.15, wspace=0.15, left=0.05, right=0.92, top=0.92, bottom=0.08)
axes = gs.subplots()

# Calculate TOP3 - ALL for detection times (using aggregated means)
time_diffs_016 = []
for cfg, _ in configs_016:
    top3_col = f'AWW_TOP3_{cfg}_mean_detection_time'
    all_col = f'AWW_ALL_{cfg}_mean_detection_time'
    diff = aggregated_data[top3_col] - aggregated_data[all_col]
    time_diffs_016.append(diff.values)

# Calculate TOP3 / ALL for local cases (using aggregated means, direct ratio)
cases_ratios_016 = []
for cfg, _ in configs_016:
    top3_col = f'AWW_TOP3_{cfg}_mean_local_cases'
    all_col = f'AWW_ALL_{cfg}_mean_local_cases'
    ratio = aggregated_data[top3_col] / aggregated_data[all_col]
    cases_ratios_016.append(ratio.values)

# Global vmin/vmax for TIME differences
all_time_diffs_016 = np.concatenate(time_diffs_016)
time_vmin_016 = np.nanmin(all_time_diffs_016)
time_vmax_016 = np.nanmax(all_time_diffs_016)

# Global vmin/vmax for CASES ratios
all_cases_ratios_016 = np.concatenate(cases_ratios_016)
cases_vmin_016 = np.nanmin(all_cases_ratios_016)
cases_vmax_016 = np.nanmax(all_cases_ratios_016)

print(f"\nTime difference range: [{time_vmin_016:.2f}, {time_vmax_016:.2f}] days")
print(f"Cases ratio range: [{cases_vmin_016:.2f}, {cases_vmax_016:.2f}]")

# TOP ROW: TOP3 - ALL Detection Time Difference
for idx, (config_name, label) in enumerate(configs_016):
    ax = axes[0, idx]
    values = time_diffs_016[idx]
    
    points = np.column_stack([R0_data, gen_time_data])
    grid_z = griddata(points, values, (R0_mesh, gen_time_mesh), method='cubic')
    
    # Clip to actual min/max
    grid_z = np.clip(grid_z, time_vmin_016, time_vmax_016)
    
    norm = Normalize(vmin=time_vmin_016, vmax=time_vmax_016)
    contour = ax.contourf(R0_mesh, gen_time_mesh, grid_z, levels=50, cmap=cmap, norm=norm)
    
    ax.set_xlim(1.5, 3.0)
    ax.set_ylim(4.0, 10.0)
    ax.tick_params(axis='both', which='major', labelsize=16)
    ax.set_yticks([4, 5, 6, 7, 8, 9, 10])
    ax.set_xticks([1.5, 2.0, 2.5, 3.0])
    
    # Labels
    if idx == 0:
        ax.set_ylabel('Generation Time (days)', fontsize=20)
    else:
        ax.set_yticklabels([])
    
    # Column titles
    ax.set_title(label, fontsize=20)

# Get position of top row to match colorbar height
top_row_bbox = axes[0, -1].get_position()
cbar_ax_top_016 = fig.add_axes([0.93, top_row_bbox.y0, 0.015, top_row_bbox.height])
sm_time_016 = ScalarMappable(norm=Normalize(vmin=time_vmin_016, vmax=time_vmax_016), cmap=cmap)
sm_time_016.set_array([])
cbar_top_016 = fig.colorbar(sm_time_016, cax=cbar_ax_top_016)
cbar_top_016.set_label('TOP3 - ALL\nDetection Time (days)', fontsize=18, labelpad=15)
cbar_top_016.ax.tick_params(labelsize=16)

# BOTTOM ROW: TOP3 / ALL Cases Ratio
for idx, (config_name, label) in enumerate(configs_016):
    ax = axes[1, idx]
    values = cases_ratios_016[idx]
    
    points = np.column_stack([R0_data, gen_time_data])
    grid_z = griddata(points, values, (R0_mesh, gen_time_mesh), method='cubic')
    
    # Clip to actual min/max
    grid_z = np.clip(grid_z, cases_vmin_016, cases_vmax_016)
    
    norm = Normalize(vmin=cases_vmin_016, vmax=cases_vmax_016)
    contour = ax.contourf(R0_mesh, gen_time_mesh, grid_z, levels=50, cmap=cmap, norm=norm)
    
    ax.set_xlim(1.5, 3.0)
    ax.set_ylim(4.0, 10.0)
    ax.tick_params(axis='both', which='major', labelsize=16)
    ax.set_yticks([4, 5, 6, 7, 8, 9, 10])
    ax.set_xticks([1.5, 2.0, 2.5, 3.0])
    
    # Labels
    if idx == 0:
        ax.set_ylabel('Generation Time (days)', fontsize=20)
    else:
        ax.set_yticklabels([])
    
    ax.set_xlabel('$R_0$', fontsize=20)

# Get position of bottom row to match colorbar height
bottom_row_bbox = axes[1, -1].get_position()
cbar_ax_bottom_016 = fig.add_axes([0.93, bottom_row_bbox.y0, 0.015, bottom_row_bbox.height])
sm_cases_016 = ScalarMappable(norm=Normalize(vmin=cases_vmin_016, vmax=cases_vmax_016), cmap=cmap)
sm_cases_016.set_array([])
cbar_bottom_016 = fig.colorbar(sm_cases_016, cax=cbar_ax_bottom_016)
cbar_bottom_016.set_label('TOP3 / ALL\nLocal Cases Ratio', fontsize=18, labelpad=15)
cbar_bottom_016.ax.tick_params(labelsize=16)

# Add overall figure title
fig.suptitle('TOP-3 vs ALL Countries AWW Detection (Aggregated Across Countries)', 
             fontsize=24, y=0.96)

plt.savefig("heatmap_top3_vs_all_aggregated_comparison.pdf", dpi=300, bbox_inches='tight')
plt.show()

# ============================================================================
# Print summary statistics
# ============================================================================
print("\n" + "="*80)
print("SUMMARY STATISTICS: TOP-3 vs ALL Countries (Aggregated Means)")
print("="*80)

for cfg, label in configs_016:
    print(f"\n{label}:")
    
    # Time difference statistics
    top3_col = f'AWW_TOP3_{cfg}_mean_detection_time'
    all_col = f'AWW_ALL_{cfg}_mean_detection_time'
    time_diff = aggregated_data[top3_col] - aggregated_data[all_col]
    
    print(f"  Detection Time Difference (TOP3 - ALL):")
    print(f"    Mean: {time_diff.mean():.2f} days")
    print(f"    Median: {time_diff.median():.2f} days")
    print(f"    Min: {time_diff.min():.2f} days")
    print(f"    Max: {time_diff.max():.2f} days")
    print(f"    Std: {time_diff.std():.2f} days")
    
    # Cases ratio statistics
    top3_cases_col = f'AWW_TOP3_{cfg}_mean_local_cases'
    all_cases_col = f'AWW_ALL_{cfg}_mean_local_cases'
    cases_ratio = aggregated_data[top3_col] / aggregated_data[all_col]
    
    print(f"  Local Cases Ratio (TOP3 / ALL):")
    print(f"    Mean: {cases_ratio.mean():.2f}")
    print(f"    Median: {cases_ratio.median():.2f}")
    print(f"    Min: {cases_ratio.min():.2f}")
    print(f"    Max: {cases_ratio.max():.2f}")
    print(f"    Std: {cases_ratio.std():.2f}")
    
    # Percentage of (R0, gen_time) combinations where TOP3 detects later than ALL
    pct_later = (time_diff > 0).sum() / len(time_diff) * 100
    print(f"  TOP3 detects LATER than ALL: {pct_later:.1f}% of (R0, gen_time) combinations")
    
    # Percentage of combinations where TOP3 has more cases than ALL
    pct_more_cases = (cases_ratio > 1).sum() / len(cases_ratio) * 100
    print(f"  TOP3 has MORE cases than ALL: {pct_more_cases:.1f}% of (R0, gen_time) combinations")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import ast
import re
from matplotlib.patches import Rectangle

# Helper function to parse arrays safely
def safe_parse_array(x):
    if pd.isna(x) or x is None:
        return np.array([])
    if isinstance(x, str):
        if x.strip() == 'Float64[]' or x.strip() == '[]':
            return np.array([])
        x = re.sub(r'Float64\[', '[', x)
        try:
            return np.array(ast.literal_eval(x))
        except:
            return np.array([])
    return x

# Load the dataset
df = pd.read_csv('pgfgleam/all_results/local/aww_t3_specific_results.csv')

print("Dataset columns:", df.columns.tolist())
print("Dataset shape:", df.shape)
print("Unique countries:", df['country'].unique())
print("Unique R0 values:", df['R0'].unique())
print("Unique gen_time values:", df['gen_time'].unique())

sampling_frac = '50pct'

# Use the mean columns directly (already computed)
df_valid = df[
    ~df[f'AWW_ALL_016_{sampling_frac}_mean_detection_time'].isna() &
    ~df[f'AWW_TOP3_016_{sampling_frac}_mean_detection_time'].isna() &
    ~df[f'AWW_ALL_016_{sampling_frac}_mean_local_cases'].isna() &
    ~df[f'AWW_TOP3_016_{sampling_frac}_mean_local_cases'].isna()
].copy()

print(f"\nAfter filtering: {len(df_valid)} rows with valid data")

# Get unique parameter combinations
param_combinations = df_valid[['R0', 'gen_time']].drop_duplicates().sort_values(['R0', 'gen_time'])
print(f"\nParameter combinations to plot: {len(param_combinations)}")
print(param_combinations)

# Define colors (turquoise for ALL, orange for TOP3)
color_all = '#E87D47'  # Turquoise for ALL countries
color_top3 = "#3431DE"  # Orange for TOP3

# Create a plot for each (R0, gen_time) combination
for idx, (_, param_row) in enumerate(param_combinations.iterrows()):
    R0_val = param_row['R0']
    gen_time_val = param_row['gen_time']
    
    print(f"\n{'='*60}")
    print(f"Plotting R0={R0_val}, gen_time={gen_time_val}")
    print(f"{'='*60}")
    
    # Filter data for this parameter combination (all countries)
    param_data = df_valid[
        (df_valid['R0'] == R0_val) & 
        (df_valid['gen_time'] == gen_time_val)
    ].copy()
    
    if len(param_data) == 0:
        print(f"No data for R0={R0_val}, gen_time={gen_time_val}")
        continue
    
    print(f"Number of countries: {len(param_data)}")
    print(f"Countries: {param_data['country'].tolist()}")
    
    # Extract mean values across all countries
    detection_times_all = param_data[f'AWW_ALL_016_{sampling_frac}_mean_detection_time'].values
    detection_times_top3 = param_data[f'AWW_TOP3_016_{sampling_frac}_mean_detection_time'].values
    local_cases_all = param_data[f'AWW_ALL_016_{sampling_frac}_mean_local_cases'].values
    local_cases_top3 = param_data[f'AWW_TOP3_016_{sampling_frac}_mean_local_cases'].values
    
    # Create figure with 2 panels
    fig, (ax_detection, ax_cases) = plt.subplots(1, 2, figsize=(20, 5))
    
    # =============== PLOT 1: Detection Times (LEFT) ===============
    y_pos_all = 1.15
    y_pos_top3 = 0.55
    
    # Calculate overall statistics
    all_det_mean = np.mean(detection_times_all)
    all_det_ci_low, all_det_ci_high = np.percentile(detection_times_all, [2.5, 97.5])
    top3_det_mean = np.mean(detection_times_top3)
    top3_det_ci_low, top3_det_ci_high = np.percentile(detection_times_top3, [2.5, 97.5])
    
    # ALL scatter
    np.random.seed(42)
    y_jitter_all = np.random.normal(y_pos_all, 0.02, size=len(detection_times_all))
    x_jitter_all = np.random.normal(detection_times_all, 0.12, size=len(detection_times_all))
    ax_detection.scatter(x_jitter_all, y_jitter_all, alpha=0.7, s=20, color=color_all,
                        edgecolors='black', linewidths=0.5, zorder=2, label='ALL countries')
    
    # ALL boxplot
    bp = ax_detection.boxplot([detection_times_all], positions=[y_pos_all], vert=False, widths=0.15,
                              showfliers=False, patch_artist=True,
                              boxprops=dict(facecolor='white', alpha=0.5, linewidth=2, edgecolor='black'),
                              medianprops=dict(color='black', linewidth=2.5),
                              whiskerprops=dict(linewidth=2, color='black'),
                              capprops=dict(linewidth=2, color='black'),
                              zorder=3)
    
    # ALL KDE
    if len(detection_times_all) > 5:
        kde = stats.gaussian_kde(detection_times_all, bw_method=0.3)
        x_range = np.linspace(max(0, detection_times_all.min() - 5), detection_times_all.max() + 5, 200)
        kde_values = kde(x_range)
        kde_scaled = kde_values / kde_values.max() * 0.3
        ax_detection.plot(x_range, kde_scaled + y_pos_all + 0.05, color=color_all, linewidth=2, zorder=4)
    
    # TOP3 scatter
    np.random.seed(52)
    y_jitter_top3 = np.random.normal(y_pos_top3, 0.02, size=len(detection_times_top3))
    x_jitter_top3 = np.random.normal(detection_times_top3, 0.12, size=len(detection_times_top3))
    ax_detection.scatter(x_jitter_top3, y_jitter_top3, alpha=0.7, s=20, color=color_top3,
                        edgecolors='black', linewidths=0.5, zorder=2, label='TOP-3 countries')
    
    # TOP3 boxplot
    bp = ax_detection.boxplot([detection_times_top3], positions=[y_pos_top3], vert=False, widths=0.15,
                              showfliers=False, patch_artist=True,
                              boxprops=dict(facecolor='white', alpha=0.5, linewidth=2, edgecolor='black'),
                              medianprops=dict(color='black', linewidth=2.5),
                              whiskerprops=dict(linewidth=2, color='black'),
                              capprops=dict(linewidth=2, color='black'),
                              zorder=3)
    
    # TOP3 KDE
    if len(detection_times_top3) > 5:
        kde = stats.gaussian_kde(detection_times_top3, bw_method=0.3)
        x_range = np.linspace(max(0, detection_times_top3.min() - 5), detection_times_top3.max() + 5, 200)
        kde_values = kde(x_range)
        kde_scaled = kde_values / kde_values.max() * 0.3
        ax_detection.plot(x_range, kde_scaled + y_pos_top3 + 0.05, color=color_top3, linewidth=2, zorder=4)
    
    # Add legend boxes for detection times
    xlim_det = ax_detection.get_xlim()
    x_legend_start = xlim_det[0] + (xlim_det[1] - xlim_det[0]) * 0.7
    box_width = (xlim_det[1] - xlim_det[0]) * 0.035
    text_offset = (xlim_det[1] - xlim_det[0]) * 0.045
    box_height = 0.1
    
    # ALL legend
    ax_detection.add_patch(Rectangle((x_legend_start, y_pos_all + 0.05+ 0.2), box_width, box_height,
                                     fc=color_all, ec='black', linewidth=2, zorder=10))
    ax_detection.text(x_legend_start + text_offset, y_pos_all + 0.1+ 0.2,
                     f"Mean: {all_det_mean:.1f}d\n95% CI: [{all_det_ci_low:.1f}, {all_det_ci_high:.1f}]",
                     fontsize=18, va='center', ha='left', zorder=10)
    
    # TOP3 legend
    ax_detection.add_patch(Rectangle((x_legend_start, y_pos_top3 + 0.05+ 0.2), box_width, box_height,
                                     fc=color_top3, ec='black', linewidth=2, zorder=10))
    ax_detection.text(x_legend_start + text_offset, y_pos_top3 + 0.1+ 0.2,
                     f"Mean: {top3_det_mean:.1f}d\n95% CI: [{top3_det_ci_low:.1f}, {top3_det_ci_high:.1f}]",
                     fontsize=18, va='center', ha='left', zorder=10)
    
    # ax_detection.set_xlabel('Detection Time (days)', fontsize=22)
    ax_detection.set_yticks([])
    ax_detection.grid(axis='x', alpha=0.3, linestyle='--', linewidth=1)
    ax_detection.set_ylim([0.4, 1.6])
    ax_detection.tick_params(axis='x', labelsize=20)
    # ax_detection.set_title(f'Detection Time\n$R_0$={R0_val}, Gen Time={gen_time_val}d', fontsize=22, pad=15)
    
    # =============== PLOT 2: Local Cases (RIGHT) ===============
    # Calculate overall statistics
    all_cases_mean = np.mean(local_cases_all)
    all_cases_ci_low, all_cases_ci_high = np.percentile(local_cases_all, [2.5, 97.5])
    top3_cases_mean = np.mean(local_cases_top3)
    top3_cases_ci_low, top3_cases_ci_high = np.percentile(local_cases_top3, [2.5, 97.5])
    
    # ALL scatter
    np.random.seed(62)
    y_jitter_all = np.random.normal(y_pos_all, 0.02, size=len(local_cases_all))
    ax_cases.scatter(local_cases_all, y_jitter_all, alpha=0.7, s=20, color=color_all,
                    edgecolors='black', linewidths=0.5, zorder=2)
    
    # ALL boxplot
    bp = ax_cases.boxplot([local_cases_all], positions=[y_pos_all], vert=False, widths=0.15,
                          showfliers=False, patch_artist=True,
                          boxprops=dict(facecolor='white', alpha=0.5, linewidth=2, edgecolor='black'),
                          medianprops=dict(color='black', linewidth=2.5),
                          whiskerprops=dict(linewidth=2, color='black'),
                          capprops=dict(linewidth=2, color='black'),
                          zorder=3)
    
    # ALL KDE
    if len(local_cases_all) > 5:
        kde = stats.gaussian_kde(local_cases_all, bw_method=0.3)
        x_range = np.linspace(max(0, local_cases_all.min() - 5), local_cases_all.max() + 5, 200)
        kde_values = kde(x_range)
        kde_scaled = kde_values / kde_values.max() * 0.3
        ax_cases.plot(x_range, kde_scaled + y_pos_all + 0.05, color=color_all, linewidth=2, zorder=4)
    
    # TOP3 scatter
    np.random.seed(72)
    y_jitter_top3 = np.random.normal(y_pos_top3, 0.02, size=len(local_cases_top3))
    ax_cases.scatter(local_cases_top3, y_jitter_top3, alpha=0.7, s=20, color=color_top3,
                    edgecolors='black', linewidths=0.5, zorder=2)
    
    # TOP3 boxplot
    bp = ax_cases.boxplot([local_cases_top3], positions=[y_pos_top3], vert=False, widths=0.15,
                          showfliers=False, patch_artist=True,
                          boxprops=dict(facecolor='white', alpha=0.5, linewidth=2, edgecolor='black'),
                          medianprops=dict(color='black', linewidth=2.5),
                          whiskerprops=dict(linewidth=2, color='black'),
                          capprops=dict(linewidth=2, color='black'),
                          zorder=3)
    
    # TOP3 KDE
    if len(local_cases_top3) > 5:
        kde = stats.gaussian_kde(local_cases_top3, bw_method=0.3)
        x_range = np.linspace(max(0, local_cases_top3.min() - 5), local_cases_top3.max() + 5, 200)
        kde_values = kde(x_range)
        kde_scaled = kde_values / kde_values.max() * 0.3
        ax_cases.plot(x_range, kde_scaled + y_pos_top3 + 0.05, color=color_top3, linewidth=2, zorder=4)
    
    # Add legend boxes for local cases
    xlim_cases = ax_cases.get_xlim()
    x_legend_start = xlim_cases[1] * 0.66
    box_width = (xlim_cases[1] - xlim_cases[0]) * 0.035
    text_offset = (xlim_cases[1] - xlim_cases[0]) * 0.045
    
    # ALL legend
    ax_cases.add_patch(Rectangle((x_legend_start, y_pos_all + 0.05 + 0.2), box_width, box_height,
                                 fc=color_all, ec='black', linewidth=2, zorder=10))
    ax_cases.text(x_legend_start + text_offset, y_pos_all + 0.1+ 0.2,
                 f"Mean: {all_cases_mean:.1f}\n95% CI: [{all_cases_ci_low:.1f}, {all_cases_ci_high:.1f}]",
                 fontsize=18, va='center', ha='left', zorder=10)
    
    # TOP3 legend
    ax_cases.add_patch(Rectangle((x_legend_start, y_pos_top3 + 0.05+ 0.2), box_width, box_height,
                                 fc=color_top3, ec='black', linewidth=2, zorder=10))
    ax_cases.text(x_legend_start + text_offset, y_pos_top3 + 0.1+ 0.2,
                 f"Mean: {top3_cases_mean:.1f}\n95% CI: [{top3_cases_ci_low:.1f}, {top3_cases_ci_high:.1f}]",
                 fontsize=18, va='center', ha='left', zorder=10)
    
    # ax_cases.set_xlabel('Local Cases at Detection', fontsize=22)
    ax_cases.set_yticks([])
    ax_cases.grid(axis='x', alpha=0.3, linestyle='--', linewidth=1)
    ax_cases.set_ylim([0.4, 1.6])
    ax_cases.tick_params(axis='x', labelsize=20)
    # ax_cases.set_title(f'Local Cases at Detection\n$R_0$={R0_val}, Gen Time={gen_time_val}d', fontsize=22, pad=15)
    
    plt.tight_layout()
    
    # Save figure
    output_filename = f'aww_t3_means_comparison_R0_{R0_val}_gentime_{gen_time_val}_samplingfrac_{sampling_frac}.pdf'
    plt.savefig(output_filename, bbox_inches='tight', dpi=300)
    plt.show()
    print(f"Saved: {output_filename}")
    
    # Print summary statistics
    print(f"\nSummary for R0={R0_val}, gen_time={gen_time_val}:")
    print(f"Number of countries: {len(param_data)}")
    print("\n--- DETECTION TIMES (across all countries) ---")
    print(f"  ALL: {all_det_mean:.2f} ({all_det_ci_low:.2f}-{all_det_ci_high:.2f}) days")
    print(f"  TOP3: {top3_det_mean:.2f} ({top3_det_ci_low:.2f}-{top3_det_ci_high:.2f}) days")
    print(f"  Difference (TOP3 - ALL): {top3_det_mean - all_det_mean:.2f} days")
    
    print("\n--- LOCAL CASES (across all countries) ---")
    print(f"  ALL: {all_cases_mean:.2f} ({all_cases_ci_low:.2f}-{all_cases_ci_high:.2f})")
    print(f"  TOP3: {top3_cases_mean:.2f} ({top3_cases_ci_low:.2f}-{top3_cases_ci_high:.2f})")
    print(f"  Ratio (TOP3 / ALL): {top3_cases_mean / all_cases_mean:.2f}")
    
    plt.close()

print("\n✓ All plots generated!")

In [ ]:
"""
Synthetic network-based epidemic importations
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar
from scipy.special import gamma as gamma_func
import seaborn as sns

sns.set_style("whitegrid")
np.random.seed(42)

# ============================================================================
# PARAMETERS
# ============================================================================

R0 = 2
generation_time = 5
r = np.log(R0) / generation_time

t_max = 150
dt = 0.1
time_points = np.arange(0, t_max, dt)

C_origin = 10

# ============================================================================
# NETWORK SIMULATION (same as before)
# ============================================================================

def simulate_network_multinode(time_points, C_O, r, n_countries=8, 
                               max_path_length=3, include_direct=True, seed=None):
    """Network simulation - returns Y(t) and Λ(t)"""
    dt = time_points[1] - time_points[0]
    
    if seed is not None:
        np.random.seed(seed)
    
    n = n_countries + 2
    origin_idx = 0
    uk_idx = n - 1
    
    countries = []
    for i in range(1, n-1):
        country = {
            'idx': i,
            'name': f'Country {i}',
            't_introduction': np.random.uniform(5, 25),
            'C_i': np.random.uniform(3, 12),
        }
        countries.append(country)
    
    M = np.zeros((n, n))
    for i in range(1, n-1):
        M[origin_idx, i] = np.random.uniform(0.00001, 0.00003)
    for i in range(1, n-1):
        M[i, uk_idx] = np.random.uniform(0.0001, 0.0005)
    for i in range(1, n-1):
        for j in range(1, n-1):
            if i != j:
                if np.random.random() < 0.4:
                    M[i, j] = np.random.uniform(0.00001, 0.00008)
    if include_direct:
        M[origin_idx, uk_idx] = 0.000005
    
    tau = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if M[i, j] > 0:
                tau[i, j] = np.random.uniform(0.5, 3.0)
    
    def find_paths_to_uk(source, current_path, all_paths, max_length):
        current = current_path[-1]
        if current == uk_idx:
            all_paths.append(current_path.copy())
            return
        if len(current_path) >= max_length + 1:
            return
        for next_node in range(n):
            if M[current, next_node] > 0 and next_node not in current_path:
                current_path.append(next_node)
                find_paths_to_uk(source, current_path, all_paths, max_length)
                current_path.pop()
    
    all_pathway_info = []
    origin_paths = []
    find_paths_to_uk(origin_idx, [origin_idx], origin_paths, max_path_length)
    for path in origin_paths:
        all_pathway_info.append({
            'path': path,
            'is_origin': True
        })
    
    for country in countries:
        country_paths = []
        find_paths_to_uk(country['idx'], [country['idx']], country_paths, max_path_length)
        for path in country_paths:
            all_pathway_info.append({
                'path': path,
                'is_origin': False,
                'country': country
            })
    
    total_lambda = np.zeros_like(time_points)
    
    for pathway_info in all_pathway_info:
        path = pathway_info['path']
        lambda_path = np.zeros_like(time_points)
        
        path_prob = 1.0
        total_delay = 0.0
        growth_during_transit = 0.0
        
        for i in range(len(path) - 1):
            from_node = path[i]
            to_node = path[i + 1]
            path_prob *= M[from_node, to_node]
            total_delay += tau[from_node, to_node]
            growth_during_transit += tau[from_node, to_node]
        
        if pathway_info['is_origin']:
            for idx, t in enumerate(time_points):
                t_effective = t - total_delay
                if t_effective >= 0:
                    I_source = C_O * np.exp(r * t_effective)
                    I_arriving = I_source * np.exp(r * growth_during_transit)
                    lambda_path[idx] = path_prob * I_arriving
        else:
            country = pathway_info['country']
            generation_delay = generation_time
            for idx, t in enumerate(time_points):
                t_effective = t - total_delay
                if t_effective >= country['t_introduction'] + generation_delay:
                    t_since_intro = t_effective - country['t_introduction'] - generation_delay
                    I_source = country['C_i'] * np.exp(r * t_since_intro)
                    I_arriving = I_source * np.exp(r * growth_during_transit)
                    lambda_path[idx] = path_prob * I_arriving
        
        total_lambda += lambda_path
    
    # Simulate Y from Λ
    cumulative = 0
    importations = []
    for lam in total_lambda:
        expected = lam * dt
        new_imports = np.random.poisson(expected)
        cumulative += new_imports
        importations.append(cumulative)
    
    return np.array(importations), total_lambda

# ============================================================================
# EXTRACT BOTH Λ(t) and Y(t) from simulations
# ============================================================================

def monte_carlo_extract_lambda_and_Y(n_sims, time_idx):
    """
    Run many simulations and extract:
    - Λ(t) values (the RATE at time t)
    - Y(t) values (the cumulative COUNT at time t)
    """
    lambda_values = []
    Y_values = []
    
    for i in range(n_sims):
        Y, lambda_t = simulate_network_multinode(
            time_points, C_origin, r, n_countries=6, max_path_length=3,
            include_direct=True, seed=None
        )
        lambda_values.append(lambda_t[time_idx])
        Y_values.append(Y[time_idx])
    
    return np.array(lambda_values), np.array(Y_values)

# ============================================================================
# ANALYSIS AT MULTIPLE TIME POINTS
# ============================================================================

n_sims = 2500

# Define time points to analyze
time_indices = [
    int(0.2 * len(time_points)),  # Early
    int(0.35 * len(time_points)), # Mid-early
    int(0.5 * len(time_points)),  # Mid
]

time_names = ['Early', 'Mid-Early', 'Mid']

print("="*80)
print("THEORETICAL CONNECTION: Λ ~ Gamma ⟹ Y ~ NegBin")
print("="*80)

for time_idx, time_name in zip(time_indices, time_names):
    t = time_points[time_idx]
    
    print(f"\n{'='*80}")
    print(f"TIME POINT: {time_name} (t={t:.1f} days)")
    print(f"{'='*80}")
    
    # ========================================================================
    # STEP 1: Extract both Λ and Y from simulations
    # ========================================================================
    
    print(f"\nRunning {n_sims} simulations to extract Λ(t) and Y(t)...")
    lambda_values, Y_values = monte_carlo_extract_lambda_and_Y(n_sims, time_idx)
    
    # ========================================================================
    # STEP 2: Fit Gamma to Λ(t)
    # ========================================================================
    
    mean_lambda = np.mean(lambda_values)
    var_lambda = np.var(lambda_values)
    
    # Gamma parameters from method of moments
    # For Gamma: E[X] = α/β, Var[X] = α/β²
    # Therefore: β = E[X]/Var[X], α = E[X] × β
    beta_lambda = mean_lambda / var_lambda
    alpha_lambda = mean_lambda * beta_lambda
    
    # Also fit using k,θ parameterization
    theta_lambda = var_lambda / mean_lambda
    k_lambda = mean_lambda / theta_lambda
    
    print(f"\nSTEP 2: Fit Gamma to Λ(t) [instantaneous rate]")
    print(f"  E[Λ] = {mean_lambda:.4f}")
    print(f"  Var[Λ] = {var_lambda:.6f}")
    print(f"  Gamma parameters: α={alpha_lambda:.2f}, β={beta_lambda:.2f}")
    
    # K-S test for Gamma fit to Λ
    ks_stat_lambda, p_lambda = stats.kstest(lambda_values,
                                             lambda x: stats.gamma.cdf(x, a=k_lambda, scale=theta_lambda))
    print(f"  K-S test: statistic={ks_stat_lambda:.6f}, p={p_lambda:.6f}")
    
    # ========================================================================
    # STEP 3: Fit NegBin to Y(t)
    # ========================================================================
    
    mean_Y = np.mean(Y_values)
    var_Y = np.var(Y_values)
    
    print(f"\nSTEP 3: Fit NegBin to Y(t) [cumulative imports]")
    print(f"  E[Y] = {mean_Y:.2f}")
    print(f"  Var[Y] = {var_Y:.2f}")
    print(f"  Var/Mean = {var_Y/mean_Y:.3f}")
    
    if var_Y > mean_Y:
        # Method of moments for NegBin
        p_fitted = mean_Y / var_Y
        r_fitted = mean_Y * p_fitted / (1 - p_fitted)
    else:
        # If not overdispersed, clamp dispersion slightly above Poisson
        # to still allow NegBin fit (avoids r → ∞ / p → 1 instability)
        var_Y_adj = mean_Y * 1.01
        p_fitted = mean_Y / var_Y_adj
        r_fitted = mean_Y * p_fitted / (1 - p_fitted)
        print(f"  Note: Data not overdispersed (Var/Mean={var_Y/mean_Y:.3f}), using adjusted variance for NegBin fit")
    
    print(f"  Fitted NegBin: r={r_fitted:.2f}, p={p_fitted:.4f}")
    print(f"  NegBin mean = {r_fitted*(1-p_fitted)/p_fitted:.2f}")
    print(f"  NegBin var = {r_fitted*(1-p_fitted)/p_fitted**2:.2f}")
    
    # K-S test
    ks_stat, p_val = stats.kstest(Y_values,
                                    lambda x: stats.nbinom.cdf(x, n=r_fitted, p=p_fitted))
    print(f"  K-S test: statistic={ks_stat:.6f}, p={p_val:.6f}")
    
    if p_val > 0.05:
        print(f"  ✓ NegBin is a good fit (K-S p > 0.05)")
    else:
        print(f"  ✗ NegBin fit is marginal (K-S p < 0.05)")

    # Fit Poisson for comparison (only for the early time point)
    if time_name == "Early":
        poisson_lambda = mean_Y
        ks_stat_pois, p_val_pois = stats.kstest(Y_values,
                                        lambda x: stats.poisson.cdf(x, mu=poisson_lambda))
        print(f"  Poisson fit: λ={poisson_lambda:.2f}")
        print(f"  K-S test: statistic={ks_stat_pois:.6f}, p={p_val_pois:.6f}")
    
    # ========================================================================
    # STEP 4: Compare theoretical prediction vs empirical fit
    # ========================================================================
    
    print(f"\nSTEP 4: Theoretical connection check")
    print(f"  If Λ ~ Gamma(α, β), then Y ~ NegBin(r=α, p=β/(1+β))")
    print(f"  Predicted from Λ: r={alpha_lambda:.2f}, p={beta_lambda/(1+beta_lambda):.4f}")
    if r_fitted is not None:
        print(f"  Fitted to Y:      r={r_fitted:.2f}, p={p_fitted:.4f}")
        print(f"  Difference: Δr={abs(alpha_lambda-r_fitted):.2f}, Δp={abs(beta_lambda/(1+beta_lambda)-p_fitted):.4f}")
    
    # ========================================================================
    # STEP 5: VISUALIZATION - Two panels
    # ========================================================================
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # =====================================================================
    # LEFT PANEL: Λ distribution with fitted Gamma
    # =====================================================================
    axes[0].hist(lambda_values, bins=30, density=True, alpha=0.7, color='blue',
                  edgecolor='black', label=f'Empirical Λ(t)')# (n={n_sims})')
    
    x_lambda = np.linspace(0, max(lambda_values)*1.2, 1000)
    gamma_pdf = stats.gamma.pdf(x_lambda, a=k_lambda, scale=theta_lambda)
    axes[0].plot(x_lambda, gamma_pdf, 'r-', linewidth=2.5,
                  label=f'Gamma Fit')#(α={alpha_lambda:.2f}, β={beta_lambda:.2f})')
    
    # axes[0].set_xlabel('Λ(t) - Instantaneous Rate', fontsize=16, fontweight='bold')
    # axes[0].set_ylabel('Probability Density', fontsize=16, fontweight='bold')
    # axes[0].set_title(f'Fit Gamma to Rate Λ(t)\n(t={t:.1f}d, K-S p={p_lambda:.3f})', 
    #                    fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=16)
    axes[0].grid(alpha=0.3)
    
    # Add text box with theoretical prediction
    # textstr = f'Theoretical prediction:\nY ~ NegBin(r={alpha_lambda:.2f},\n            p={beta_lambda/(1+beta_lambda):.3f})'
    # props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    # axes[0].text(0.98, 0.97, textstr, transform=axes[0].transAxes, fontsize=10,
    #             verticalalignment='top', horizontalalignment='right', bbox=props)
    
    # =====================================================================
    # RIGHT PANEL: Y distribution with fitted NegBin (+ Poisson for Early)
    # =====================================================================
    bins_Y = np.arange(Y_values.min()-0.5, Y_values.max()+1.5, 
                       max(1, int((Y_values.max() - Y_values.min()) / 30)))
    axes[1].hist(Y_values, bins=bins_Y, density=True, alpha=0.7, color='green',
                  edgecolor='black', label=f'Empirical Y(t)')

    x_Y = np.arange(0, int(Y_values.max())+1)
    
    # Always plot NegBin
    nb_pmf = stats.nbinom.pmf(x_Y, n=r_fitted, p=p_fitted)
    axes[1].plot(x_Y, nb_pmf, 'r-', linewidth=2.5, marker='o', markersize=3,
                  label='Negative Binomial Fit')
    
    # Also plot Poisson for Early time point
    if time_name == "Early":
        poisson_lambda = mean_Y
        poisson_pmf = stats.poisson.pmf(x_Y, mu=poisson_lambda)
        axes[1].plot(x_Y, poisson_pmf, 'b--', linewidth=2.5, marker='s', markersize=3,
                      label='Poisson Fit')
    
    axes[1].legend(fontsize=16)
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'theoretical_connection_{time_name.lower().replace("-","")}_t{int(t)}.pdf', 
                 dpi=300, bbox_inches='tight')
    print(f"\nSaved: theoretical_connection_{time_name.lower().replace('-','')}_t{int(t)}.pdf")
    plt.show()
    plt.close()
